<a href="https://colab.research.google.com/github/Rakhayeva/multilingual-persuasion-detection/blob/main/05_transformer_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 05: Transformer Experiments

This notebook fine-tunes XLM-RoBERTa for persuasion detection
and evaluates cross-lingual transfer to Kazakh and Russian.

## Why XLM-RoBERTa?
XLM-RoBERTa is a multilingual transformer pretrained on 100 languages
including Russian and Kazakh. Unlike TF-IDF, it learns contextual
representations that may partially capture cross-lingual patterns,
making it a stronger baseline for cross-lingual transfer (H1).

## Experiments
- **Baseline:** Train on EN, Test on EN (in-language)
- **H1:** Train on EN, Test on KZ and RU proxy sets
- Compare results with classical ML from notebook 04

In [1]:
#Path
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/NU/SEDS/multilingual-persuasion-detection'

Mounted at /content/drive


In [2]:
# Libraries
import os
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import (AutoTokenizer,
                           AutoModelForSequenceClassification,
                           TrainingArguments,
                           Trainer)
from sklearn.metrics import f1_score, classification_report

In [3]:
# Check GPU availability
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")

GPU available: True
GPU device: Tesla T4


In [4]:
# Load datasets
df_train   = pd.read_csv(f'{BASE}/data/processed/train.csv')
df_val     = pd.read_csv(f'{BASE}/data/processed/val.csv')
df_test_en = pd.read_csv(f'{BASE}/data/processed/test_en.csv')
df_test_kz = pd.read_csv(f'{BASE}/data/processed/test_kz.csv')
df_test_ru = pd.read_csv(f'{BASE}/data/processed/test_ru.csv')

print("\nLoaded:")
print(f"  Train:   {len(df_train)}")
print(f"  Val:     {len(df_val)}")
print(f"  Test EN: {len(df_test_en)}")
print(f"  Test KZ: {len(df_test_kz)} (proxy)")
print(f"  Test RU: {len(df_test_ru)} (proxy)")


Loaded:
  Train:   746
  Val:     160
  Test EN: 160
  Test KZ: 101 (proxy)
  Test RU: 103 (proxy)


## Tokenization

In [5]:
# Load tokenizer
# XLM-RoBERTa-base is pretrained on 100 languages including Russian and Kazakh.
# max_length=256 covers most EN paragraphs and all KZ/RU texts
# (mean word count: EN=270, KZ=40, RU=35)

MODEL_NAME = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer loaded: {MODEL_NAME}")

def tokenize(examples):
    return tokenizer(
        examples['text_clean'],
        truncation=True,
        max_length=256,
        padding='max_length'
    )

def to_dataset(df):
    """
    Converts a DataFrame to a HuggingFace Dataset.
    Renames has_persuasion to labels (required by Trainer).
    """
    ds = Dataset.from_pandas(
        df[['text_clean', 'has_persuasion']]
          .rename(columns={'has_persuasion': 'labels'})
          .reset_index(drop=True)
    )
    ds = ds.map(tokenize, batched=True)
    ds = ds.remove_columns(['text_clean'])
    ds.set_format('torch')
    return ds

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded: xlm-roberta-base


In [6]:
# Tokenize all splits
print("Tokenizing datasets...")
ds_train   = to_dataset(df_train.fillna({'text_clean': ''}))
ds_val     = to_dataset(df_val.fillna({'text_clean': ''}))
ds_test_en = to_dataset(df_test_en.fillna({'text_clean': ''}))
ds_test_kz = to_dataset(df_test_kz.fillna({'text_clean': ''}))
ds_test_ru = to_dataset(df_test_ru.fillna({'text_clean': ''}))

print("Done!")
print(f"Train dataset: {ds_train}")

Tokenizing datasets...


Map:   0%|          | 0/746 [00:00<?, ? examples/s]

Map:   0%|          | 0/160 [00:00<?, ? examples/s]

Map:   0%|          | 0/160 [00:00<?, ? examples/s]

Map:   0%|          | 0/101 [00:00<?, ? examples/s]

Map:   0%|          | 0/103 [00:00<?, ? examples/s]

Done!
Train dataset: Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 746
})


## Model Training and Evaluation

In [9]:
# Metrics function
# F1-macro is the primary metric: treats both classes equally
# and is standard for persuasion detection tasks (SemEval).

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'f1_macro': f1_score(labels, preds,
                              average='macro', zero_division=0),
        'accuracy': (preds == labels).mean()
    }

# Training function
def train_xlmr(train_ds, val_ds, name):
    """
    Fine-tunes XLM-RoBERTa on the given training dataset.
    Saves best model checkpoint based on F1-macro on validation set.
    """
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2)

    args = TrainingArguments(
        output_dir=f'/tmp/checkpoints/{name}',
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        warmup_ratio=0.1,
        weight_decay=0.01,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='f1_macro',
        fp16=torch.cuda.is_available(),
        report_to='none',           # disable wandb logging
        logging_steps=50
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics
    )

    trainer.train()
    return model, trainer

# Evaluation function
def evaluate_model(model, test_ds, test_df, label=''):
    """
    Evaluates model on a test dataset.
    Returns F1-macro and prints classification report.
    """
    trainer = Trainer(model=model, compute_metrics=compute_metrics)
    results = trainer.evaluate(test_ds)
    f1 = results['eval_f1_macro']

    # Detailed classification report
    preds = np.argmax(
        trainer.predict(test_ds).predictions, axis=-1)
    print(f"\n{'='*50}")
    print(f"  {label}")
    print(f"{'='*50}")
    print(classification_report(
        test_df['has_persuasion'],
        preds,
        target_names=['Not manipulative', 'Manipulative'],
        zero_division=0
    ))
    return round(f1, 3)

In [10]:
# Train model on English
print("Training XLM-RoBERTa on English data...")
print("This will take approximately 10-15 minutes on T4 GPU.\n")

model_en, trainer_en = train_xlmr(ds_train, ds_val, 'xlmr_en')
print("Training complete!")

Training XLM-RoBERTa on English data...
This will take approximately 10-15 minutes on T4 GPU.



Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,F1 Macro,Accuracy
1,No log,0.497604,0.623782,0.693750
2,0.641871,0.473842,0.818175,0.818750
3,0.500601,0.437028,0.812030,0.812500


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Training complete!


The model selected the epoch 2 checkpoint as the best. The F1-macro of 0.818 in the validation set is significantly better than the baseline LR (0.761). We're now evaluating this on the test sets.

In [11]:
# Evaluate on all test sets
# We evaluate the EN-trained model on three test sets:
# EN: in-language performance (upper bound)
# KZ: cross-lingual transfer to Kazakh proxy (H1)
# RU: cross-lingual transfer to Russian proxy (H1)

trans_results = []

for ds_test, df_test, test_lang, hypothesis in [
    (ds_test_en, df_test_en, 'EN',        'Baseline'),
    (ds_test_kz, df_test_kz, 'KZ(proxy)', 'H1'),
    (ds_test_ru, df_test_ru, 'RU(proxy)', 'H1'),
]:
    label = f"XLM-R | Train: EN, Test: {test_lang}"
    f1 = evaluate_model(model_en, ds_test, df_test, label)

    trans_results.append({
        'Model':      'XLM-RoBERTa',
        'Train':      'EN',
        'Test':       test_lang,
        'Hypothesis': hypothesis,
        'F1-macro':   f1
    })

# ── Results table ─────────────────────────────────────────────────────────────
df_trans = pd.DataFrame(trans_results)
print("\nXLM-RoBERTa Results:")
print(df_trans.to_string(index=False))


  XLM-R | Train: EN, Test: EN
                  precision    recall  f1-score   support

Not manipulative       0.67      0.88      0.76        66
    Manipulative       0.89      0.70      0.79        94

        accuracy                           0.78       160
       macro avg       0.78      0.79      0.77       160
    weighted avg       0.80      0.78      0.78       160




  XLM-R | Train: EN, Test: KZ(proxy)
                  precision    recall  f1-score   support

Not manipulative       0.50      0.98      0.66        51
    Manipulative       0.00      0.00      0.00        50

        accuracy                           0.50       101
       macro avg       0.25      0.49      0.33       101
    weighted avg       0.25      0.50      0.33       101




  XLM-R | Train: EN, Test: RU(proxy)
                  precision    recall  f1-score   support

Not manipulative       0.48      1.00      0.64        49
    Manipulative       0.00      0.00      0.00        54

        accuracy                           0.48       103
       macro avg       0.24      0.50      0.32       103
    weighted avg       0.23      0.48      0.31       103


XLM-RoBERTa Results:
      Model Train      Test Hypothesis  F1-macro
XLM-RoBERTa    EN        EN   Baseline     0.774
XLM-RoBERTa    EN KZ(proxy)         H1     0.331
XLM-RoBERTa    EN RU(proxy)         H1     0.322


**EN to EN (in-language baseline):**
XLM-RoBERTa achieves F1-macro of 0.774, outperforming the best
baseline model (LR: 0.761, SVM: 0.759). The improvement is modest
but consistent, showing that contextual multilingual representations
provide a small advantage over TF-IDF features for English
persuasion detection.

**EN to KZ and EN to RU (cross-lingual transfer, H1):**
Despite XLM-RoBERTa being pretrained on 100 languages including
Russian and Kazakh, cross-lingual transfer fails completely.
Both KZ (F1: 0.331) and RU (F1: 0.322) show degenerate prediction:
the model assigns every text to class 0 (not manipulative),
identical to the classical ML baseline behavior.

This is a critical finding: even a powerful multilingual transformer
pretrained on the target languages cannot bridge the gap between
English persuasion detection and Kazakh/Russian fake news detection.
The failure is not due to language mismatch alone, but also due to
task mismatch: the model learned persuasion-specific features in
English that do not correspond to manipulation signals in the
proxy task.

**XLM-RoBERTa vs Classical ML:**
The transformer provides no advantage for cross-lingual transfer
in this setting. Both approaches fail equally on KZ and RU,
suggesting the bottleneck is the task definition difference,
not the model architecture.

In [12]:
# Save results and model
os.makedirs(f'{BASE}/results/models', exist_ok=True)

# Save metrics
df_trans.to_csv(f'{BASE}/results/metrics_transformer.csv', index=False)
print("Saved: results/metrics_transformer.csv")

# Save fine-tuned model to Drive for SHAP analysis
model_en.save_pretrained(f'{BASE}/results/models/xlmr_en_model')
tokenizer.save_pretrained(f'{BASE}/results/models/xlmr_en_model')
print("Saved: results/models/xlmr_en_model/")

print("\nAll outputs saved to Google Drive.")

Saved: results/metrics_transformer.csv


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved: results/models/xlmr_en_model/

All outputs saved to Google Drive.


## Summary

| Model | Train | Test | F1-macro | vs Baseline LR |
|-------|-------|------|----------|----------------|
| XLM-R | EN | EN | 0.774 | +0.013 |
| XLM-R | EN | KZ (proxy) | 0.331 | -0.005 |
| XLM-R | EN | RU (proxy) | 0.322 | -0.009 |

**Key finding:** XLM-RoBERTa marginally outperforms classical ML
on English in-language detection, but provides no cross-lingual
transfer advantage to Kazakh and Russian. The cross-lingual gap
documented in [notebook 04](https://colab.research.google.com/drive/1tqHlouKHNJYKNrciYk3qfbQ_idTQIHfD#scrollTo=gsqRHvk03POL?usp=sharing) persists even with a multilingual
transformer, confirming H1 and strengthening the case for
language-specific annotated resources.

**Next:** Download outputs from Kaggle, upload to Google Drive,
then proceed to [`06_shap_explainability.ipynb`](https://colab.research.google.com/drive/1jpiAGpm5Dmfh2s56I1yFwMynhXmwm9Mw?usp=sharing) in Colab.